# FBref

## Ids times

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

def extrair_ids_com_genero():
    # Inicializa Chrome com bypass de proteção (Cloudflare)
    options = uc.ChromeOptions()
    driver = uc.Chrome(options=options, version_main=146) 
    
    url = "https://fbref.com/en/country/clubs/BRA/Brazil-Football-Clubs"
    
    try:
        print(f"🚀 Acessando FBRef...")
        driver.get(url)
        
        # Delay para garantir resolução de challenge + carregamento completo do DOM
        time.sleep(15) 
        
        # Parse do HTML já renderizado (inclui conteúdo dinâmico)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # Percorre todas as linhas de tabela
        rows = soup.find_all('tr')
        
        clubes = []
        for row in rows:
            # Extrai link do time (padrão consistente de squads)
            link = row.find('a', href=re.compile(r'/en/squads/[a-f0-9]{8}/'))
            
            # Extrai célula de gênero via atributo semântico
            gender_cell = row.find('td', {'data-stat': 'gender'})
            
            # Garante que a linha contém os dados esperados
            if link and gender_cell:
                href = link['href']
                time_id = href.split('/')[3]  # ID canônico do FBref
                
                nome_time = link.get_text(strip=True)
                genero = gender_cell.get_text(strip=True)
                
                # Regra de negócio: filtrar apenas times masculinos
                if genero == "M":
                    clubes.append({
                        'nome_fbref': nome_time,
                        'fbref_id': time_id,
                        'genero': genero
                    })

        # Deduplicação por ID (defensivo contra inconsistências do HTML)
        df = pd.DataFrame(clubes).drop_duplicates(subset=['fbref_id'])
        
        return df

    finally:
        # Liberação garantida do recurso (evita processos zumbis)
        driver.quit()


# Execução isolada da extração
df_ids_limpo = extrair_ids_com_genero()

# Persistência em CSV com encoding compatível com Excel (BOM)
df_ids_limpo.to_csv(
    'csv//ids_times_fbref_masculino.csv',
    index=False,
    sep=';',
    encoding='utf-8-sig'
)

# Feedback simples de execução
print(f"✅ Extração concluída. {len(df_ids_limpo)} times masculinos encontrados.")

In [ ]:
df_ids_limpo[df_ids_limpo['fbref_id'] == '7a1064a2']

## Mapeamento times

In [ ]:
import pandas as pd
from thefuzz import fuzz
from thefuzz import process
import re

# 1. Carregar os arquivos
# Usei o caminho que você forneceu no código anterior
df_meus_times = pd.read_csv('csv//times_2025.csv', sep=';')
df_fbref = pd.read_csv('csv//ids_times_fbref_masculino.csv', sep=';')

# Limpeza inicial: remove espaços extras que podem vir do CSV ou do Scraping
df_meus_times['nome'] = df_meus_times['nome'].str.strip()
df_fbref['nome_fbref'] = df_fbref['nome_fbref'].str.strip()

def limpar_nome_time(nome):
    """Remove prefixos e sufixos comuns para melhorar o match"""
    if not nome: return ""
    nome = nome.upper()
    # Remove SC, FC, EC, SAF, FR, Clube, Esporte, etc.
    termos = [r'\bSC\b', r'\bFC\b', r'\bEC\b', r'\bSAF\b', r'\bFR\b', 
              r'\bCLUBE\b', r'\bESPORTE\b', r'\bFUTEBOL\b', r'\bASSOCIACAO\b']
    for termo in termos:
        nome = re.sub(termo, '', nome)
    return nome.strip()

# Criar lista de nomes do FBRef para comparação
lista_fbref = df_fbref['nome_fbref'].tolist()

def mapear_times_robusto(nome_meu_banco, escolhas_fbref):
    # Tenta o match exato primeiro (mais rápido e seguro)
    if nome_meu_banco in escolhas_fbref:
        return nome_meu_banco
    
    # Se não for exato, tenta o fuzzy matching
    # Reduzimos o score para 60 para pegar variações como "SC Corinthians Paulista"
    resultado = process.extractOne(nome_meu_banco, escolhas_fbref, scorer=fuzz.token_sort_ratio)
    
    if resultado and resultado[1] >= 65:
        return resultado[0]
    
    # Última tentativa: limpa os nomes e tenta de novo
    nome_limpo = limpar_nome_time(nome_meu_banco)
    escolhas_limpas = [limpar_nome_time(x) for x in escolhas_fbref]
    
    resultado_limpo = process.extractOne(nome_limpo, escolhas_limpas, scorer=fuzz.token_sort_ratio)
    
    if resultado_limpo and resultado_limpo[1] >= 80:
        # Retorna o nome original do FBRef usando o índice do match limpo
        idx = escolhas_limpas.index(resultado_limpo[0])
        return escolhas_fbref[idx]

    return None

print("🔄 Iniciando mapeamento robusto...")

# 2. Aplicar o mapeamento
df_meus_times['nome_time_fbref'] = df_meus_times['nome'].apply(lambda x: mapear_times_robusto(x, lista_fbref))

# 3. Merge para trazer o fbref_id
df_final = pd.merge(
    df_meus_times, 
    df_fbref[['nome_fbref', 'fbref_id']], 
    left_on='nome_time_fbref', 
    right_on='nome_fbref', 
    how='left'
)

# 4. Organizar colunas
df_final = df_final[['nome', 'nome_time_fbref', 'fbref_id']]
df_final.columns = ['nome_time', 'nome_time_fbref', 'id_fbref']

# 5. Salvar
df_final.to_csv('csv//mapeamento_times.csv', index=False, sep=';', encoding='utf-8-sig')

print("✅ Mapeamento concluído!")
print(df_final.head(15))

In [ ]:
df_final.shape

In [ ]:
df_final.isna().sum()

## Extração de Jogadores (2025)

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import re

# Carrega mapeamento previamente gerado (fonte de verdade para IDs do FBref)
df_mapeamento = pd.read_csv('csv//mapeamento_times.csv', sep=';')

# Remove registros inválidos (IDs ausentes inviabilizam construção da URL)
df_mapeamento = df_mapeamento.dropna(subset=['id_fbref'])


def extrair_jogadores_2025():
    # Inicializa driver com bypass de proteção anti-bot
    options = uc.ChromeOptions()
    # options.add_argument('--headless')  # Evitar headless em fase inicial (Cloudflare mais restritivo)
    
    driver = uc.Chrome(options=options, version_main=146)
    
    dados_totais = []

    try:
        # Iteração sequencial: necessária para respeitar rate limit do FBref
        for index, row in df_mapeamento.iterrows():
            nome_original = row['nome_time']
            id_fbref = row['id_fbref']
            
            # Construção determinística da URL por temporada
            url_2025 = f"https://fbref.com/en/squads/{id_fbref}/2025/Stats"
            
            print(f"⏳ ({index+1}/{len(df_mapeamento)}) Acessando 2025 para: {nome_original}...")
            
            driver.get(url_2025)
            
            # Delay crítico para evitar bloqueio (HTTP 429 / challenge)
            time.sleep(15) 
            
            # Parse do DOM completo (após renderização JS)
            soup = BeautifulSoup(driver.page_source, "html.parser")
            
            # Busca resiliente pela tabela de estatísticas padrão
            # ID pode variar (ex: stats_standard_9), então usamos regex
            tabela_jogadores = soup.find('table', {
                'id': re.compile(r'stats_standard_')
            })
            
            if tabela_jogadores:
                tbody = tabela_jogadores.find('tbody')
                
                # Proteção contra HTML inconsistente
                if not tbody:
                    print(f"⚠️ {nome_original}: tabela sem tbody")
                    continue
                
                rows = tbody.find_all('tr')
                
                for r in rows:
                    # Coluna de jogador fica no header da linha (padrão FBref)
                    col_jogador = r.find('th', {'data-stat': 'player'})
                    
                    if col_jogador:
                        nome_jogador = col_jogador.get_text(strip=True)
                        
                        # Nem todos os jogadores têm link (ex: agregados)
                        link_tag = col_jogador.find('a')
                        link_jogador = link_tag['href'] if link_tag else ''
                        
                        dados_totais.append({
                            'time_banco': nome_original,
                            'id_fbref_time': id_fbref,
                            'nome_jogador': nome_jogador,
                            'url_fbref_jogador': f"https://fbref.com{link_jogador}" if link_jogador else '',
                            'temporada': '2025'
                        })
                
                print(f"✅ {nome_original}: jogadores extraídos")
            
            else:
                # Possíveis causas: time sem dados na temporada, bloqueio ou mudança de layout
                print(f"❌ {nome_original}: tabela de 2025 não encontrada")

        # Estrutura final pronta para persistência/ETL
        return pd.DataFrame(dados_totais)

    finally:
        # Garante encerramento do driver mesmo em erro
        driver.quit()


# Execução isolada
df_jogadores_2025 = extrair_jogadores_2025()

# Persistência apenas se houver dados (evita gerar arquivo vazio)
if not df_jogadores_2025.empty:
    df_jogadores_2025.to_csv(
        'csv//jogadores_extraidos_2025.csv',
        index=False,
        sep=';',
        encoding='utf-8-sig'  # Compatível com Excel
    )
    
    print(f"\n🚀 Finalizado! Total de {len(df_jogadores_2025)} jogadores salvos.")

## Estatísticas Completas (2025)

In [15]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os

# Carregar seu mapeamento
df_map = pd.read_csv('csv//mapeamento_times.csv', sep=';').dropna(subset=['id_fbref'])


def traduzir_posicao(pos_en):
    if not pos_en:
        return "N/I" # Não informado
    
    # Dicionário de tradução
    mapa = {
        'GK': 'Goleiro',
        'DF': 'Defensor',
        'MF': 'Meio-campista',
        'FW': 'Atacante',
        'FB': 'Lateral',
        'LB': 'Lateral Esquerdo',
        'RB': 'Lateral Direito',
        'CB': 'Zagueiro'
    }
    
    # Divide por vírgula (caso tenha mais de uma posição), traduz e junta de novo
    partes = pos_en.split(',')
    partes_traduzidas = [mapa.get(p.strip(), p.strip()) for p in partes]
    
    return ', '.join(partes_traduzidas)

def extrair_stats_completas_portugues_2025():
    options = uc.ChromeOptions()
    # options.add_argument('--headless') 
    driver = uc.Chrome(options=options, version_main=146)
    
    lista_jogadores = []

    try:
        for index, row in df_map.iterrows():
            time_nome = row['nome_time']
            fbref_id = row['id_fbref']
            
            # URL forçada para a temporada 2025
            url = f"https://fbref.com/en/squads/{fbref_id}/2025/Stats"
            
            print(f"📊 ({index+1}/{len(df_map)}) Extraindo: {time_nome}...")
            
            driver.get(url)
            # 20 segundos é o tempo de segurança recomendado para o FBRef
            time.sleep(20) 
            
            soup = BeautifulSoup(driver.page_source, "html.parser")
            
            # Localiza a tabela de estatísticas padrão
            table = soup.find('table', {'id': re.compile(r'stats_standard_')})
            
            if not table:
                print(f"⚠️ Dados de 2025 ainda não disponíveis para {time_nome}.")
                continue

            rows = table.find('tbody').find_all('tr')
            
            for r in rows:
                if r.get('class') and 'spacer' in r.get('class'):
                    continue
                
                # Função auxiliar para pegar o texto da célula pelo data-stat
                def get_val(stat_name):
                    cell = r.find(['td', 'th'], {'data-stat': stat_name})
                    return cell.get_text(strip=True) if cell else "0"

                # Mapeamento traduzido
                stats = {
                    'time': time_nome,
                    'nome': get_val('player'),
                    'nacionalidade': get_val('nationality'),
                    'posicao': traduzir_posicao(get_val('position')),
                    'idade': get_val('age'),
                    'qt_jogos': get_val('games'),
                    'qt_jogos_titular': get_val('games_starts'),
                    'minutos_jogados': get_val('minutes'),
                    'minutos_jogados_divid_90': get_val('minutes_90s'),
                    'gols': get_val('goals'),
                    'assistencias': get_val('assists'),
                    'participacoes_gols': get_val('goals_assists'),
                    'gols_nao_penalti': get_val('goals_pens'),
                    'gols_penalti': get_val('pens_made'),
                    'chutes_penalti': get_val('pens_att'),
                    'cartoes_amarelo': get_val('cards_yellow'),
                    'cartoes_vermelho': get_val('cards_red'),
                    'gols_90': get_val('goals_per90'),
                    'assistencias_90': get_val('assists_per90'),
                    'participacoes_gols_90': get_val('goals_assists_per90'),
                    'gols_nao_penalti_90': get_val('goals_pens_per90'),
                    'participacoes_gols_e_penalti_90': get_val('goals_assists_pens_per90')
                }
                
                if stats['nome']:
                    lista_jogadores.append(stats)

            print(f"✅ {time_nome} processado com sucesso.")

        return pd.DataFrame(lista_jogadores)

    finally:
        driver.quit()

# Execução
df_stats_traduzido = extrair_stats_completas_portugues_2025()

if not df_stats_traduzido.empty:
    if not os.path.exists('csv'): os.makedirs('csv')
    
    output_path = 'csv//stats_jogadores_2025.csv'
    df_stats_traduzido.to_csv(output_path, index=False, sep=';', encoding='utf-8-sig')
    
    print(f"\n🚀 Pronto! Arquivo salvo em: {output_path}")
    print(f"Colunas geradas: {list(df_stats_traduzido.columns)}")
else:
    print("\n❌ Nenhuma estatística coletada. Verifique os logs acima.")

📊 (2/31) Extraindo: Botafogo-SP...
✅ Botafogo-SP processado com sucesso.
📊 (3/31) Extraindo: Corinthians...
✅ Corinthians processado com sucesso.
📊 (4/31) Extraindo: Guarani...
⚠️ Dados de 2025 ainda não disponíveis para Guarani.
📊 (6/31) Extraindo: Mirassol...
✅ Mirassol processado com sucesso.
📊 (8/31) Extraindo: Novorizontino...
✅ Novorizontino processado com sucesso.
📊 (9/31) Extraindo: Palmeiras...
✅ Palmeiras processado com sucesso.
📊 (12/31) Extraindo: RB Bragantino...
✅ RB Bragantino processado com sucesso.
📊 (13/31) Extraindo: Santos...
✅ Santos processado com sucesso.
📊 (14/31) Extraindo: São Bernardo...
⚠️ Dados de 2025 ainda não disponíveis para São Bernardo.
📊 (15/31) Extraindo: São Paulo...
✅ São Paulo processado com sucesso.
📊 (16/31) Extraindo: Velo Clube...
✅ Velo Clube processado com sucesso.
📊 (18/31) Extraindo: Athletico Paranaense...
✅ Athletico Paranaense processado com sucesso.
📊 (22/31) Extraindo: Coritiba...
✅ Coritiba processado com sucesso.
📊 (23/31) Extraind

## Insert no banco de dados MySQL

### map_campeonatos

In [19]:
map_campeonatos = {
    # Paulista
    'Palmeiras': 1,
    'Corinthians': 1,
    'Santos': 1,
    'São Paulo': 1,
    'Mirassol': 1,
    'Novorizontino': 1,
    'Botafogo-SP': 1,
    'RB Bragantino': 1,
    'Velo Clube': 1,
    'São Bernardo': 1,

    # Paranaense
    'Athletico Paranaense': 2,
    'Coritiba': 2,
    'Operario Ferroviario': 2,

    # Gaúcho
    'Internacional': 3,
    'Juventude': 3,

    # Mineiro
    'Atlético Mineiro': 4,
    'Cruzeiro': 4,
    'America-MG': 4,
    'Athletic Club': 4,
    'Villa Nova': 4,

    # Carioca
    'Flamengo': 5,
    'Fluminense': 5,
    'Vasco da Gama': 5,
    'Botafogo': 5,
    'Volta Redonda': 5
}

### limpar_dataframe_jogadores

In [26]:
def limpar_dataframe_jogadores(df, map_campeonatos):
    """
    Limpa e padroniza DataFrame de jogadores para carga no MySQL
    """

    # Campeonato Id
    # Mapeia o time para o campeonato correspondente
    df['campeonato_id'] = df['time'].map(map_campeonatos)

    # Validação de times não mapeados (evita NULL em coluna obrigatória)
    faltantes = df[df['campeonato_id'].isnull()]['time'].unique()
    if len(faltantes) > 0:
        print("⚠️ Times sem campeonato mapeado:")
        print(faltantes)

    # Remover Linhas Inválidas
    # Remove registros que não representam jogadores (linhas agregadas do scraping)
    df = df[df['nome'].notna()]
    df = df[df['nome'] != '0']
    df = df[df['nome'] != 0]

    # Tratamento De Colunas Inteiras
    # Nessas colunas a vírgula representa milhar (ex: 2,167 → 2167)
    colunas_int = [
        'idade', 'qt_jogos', 'qt_jogos_titular', 'minutos_jogados',
        'gols', 'assistencias', 'participacoes_gols',
        'gols_nao_penalti', 'gols_penalti', 'chutes_penalti',
        'cartoes_amarelo', 'cartoes_vermelho'
    ]

    for col in colunas_int:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(',', '', regex=False)  # remove separador de milhar
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')  # força tipo numérico

    # Tratamento De Colunas Decimais
    # Essas colunas já vêm no padrão correto (ponto como decimal)
    colunas_decimal = [
        'minutos_jogados_divid_90',
        'gols_90', 'assistencias_90', 'participacoes_gols_90',
        'gols_nao_penalti_90', 'participacoes_gols_e_penalti_90'
    ]

    for col in colunas_decimal:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Tratamento De Strings
    # Remove ruídos e padroniza campos textuais
    if 'nacionalidade' in df.columns:
        df['nacionalidade'] = (
            df['nacionalidade']
            .astype(str)
            .str[-3:]  # mantém apenas código do país (ex: BRA)
        )

    df['nome'] = df['nome'].astype(str).str.strip()
    df['time'] = df['time'].astype(str).str.strip()
    df['posicao'] = df['posicao'].astype(str).str.strip()

    # Remover Duplicados
    # Evita inserir o mesmo jogador repetido
    df = df.drop_duplicates()

    # Ajuste Para MySQL
    # Converte NaN para None (necessário para inserção correta no banco)
    df = df.where(pd.notnull(df), None)

    # Resetar Índice
    # Mantém o DataFrame consistente após filtros
    df = df.reset_index(drop=True)

    # Log Final
    print(f"✅ Após limpeza: {len(df)} registros válidos")

    # Debug útil para validar correção de minutos
    if 'minutos_jogados' in df.columns:
        print(df[['minutos_jogados']].head())
        print(df['minutos_jogados'].dtype)

    return df

In [ ]:
# def limpar_dataframe_jogadores(df, map_campeonatos):
#     """
#     Limpa e padroniza DataFrame de jogadores para carga no MySQL
#     """

#     # CAMPEONATO_ID
#     df['campeonato_id'] = df['time'].map(map_campeonatos)

#     faltantes = df[df['campeonato_id'].isnull()]['time'].unique()
#     if len(faltantes) > 0:
#         print("⚠️ Times sem campeonato mapeado:")
#         print(faltantes)


#     # REMOVER LINHAS INVÁLIDAS
#     df = df[df['nome'].notna()]
#     df = df[df['nome'] != '0']
#     df = df[df['nome'] != 0]


#     # NUMÉRICOS
#     colunas_numericas = [
#         'idade', 'qt_jogos', 'qt_jogos_titular', 'minutos_jogados',
#         'minutos_jogados_divid_90', 'gols', 'assistencias',
#         'participacoes_gols', 'gols_nao_penalti', 'gols_penalti',
#         'chutes_penalti', 'cartoes_amarelo', 'cartoes_vermelho',
#         'gols_90', 'assistencias_90', 'participacoes_gols_90',
#         'gols_nao_penalti_90', 'participacoes_gols_e_penalti_90'
#     ]

#     for col in colunas_numericas:
#         if col in df.columns:
#             df[col] = (
#                 df[col]
#                 .astype(str)
#                 .str.replace('.', '', regex=False)   # remove milhar
#                 .str.replace(',', '.', regex=False)  # corrige decimal
#             )
#             df[col] = pd.to_numeric(df[col], errors='coerce')


#     # STRINGS
#     if 'nacionalidade' in df.columns:
#         df['nacionalidade'] = (
#             df['nacionalidade']
#             .astype(str)
#             .str[-3:]
#         )

#     df['nome'] = df['nome'].astype(str).str.strip()
#     df['time'] = df['time'].astype(str).str.strip()
#     df['posicao'] = df['posicao'].astype(str).str.strip()


#     # REMOVE DUPLICADOS
#     df = df.drop_duplicates()

#     # NULL PARA MYSQL
#     df = df.where(pd.notnull(df), None)

#     # FINAL
#     df = df.reset_index(drop=True)

#     print(f"✅ Após limpeza: {len(df)} registros válidos")

#     # debug útil
#     if 'minutos_jogados' in df.columns:
#         print(df[['minutos_jogados']].head())
#         print(df['minutos_jogados'].dtype)

#     return df

### tratar_valor_mysql

In [24]:
def tratar_valor_mysql(valor):
    if pd.isna(valor):
        return None
    return valor

### Insert

In [27]:
import pandas as pd
import mysql.connector
import os


df = pd.read_csv('csv//stats_jogadores_2025.csv', sep=';')
df = limpar_dataframe_jogadores(df, map_campeonatos)

# CONEXÃO
# Mudar para arquivo .env para produção
DB_HOST     = "localhost"
DB_PORT     = 3310
DB_USER     = "admin"
DB_PASSWORD = "admin"
DB_DATABASE = "n8n"

try:
    conn = mysql.connector.connect(
        host=DB_HOST,
        port=DB_PORT,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_DATABASE 
    )

    cursor = conn.cursor()

   
    # DROP e CREATE TABLE   
    cursor.execute("DROP TABLE IF EXISTS stats_jogadores_2025")
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS stats_jogadores_2025 (
        id                              INT AUTO_INCREMENT PRIMARY KEY,
        time                            VARCHAR(100) NOT NULL,
        campeonato_id                   INT NOT NULL,
        nome                            VARCHAR(150) NOT NULL,
        nacionalidade                   VARCHAR(50),
        posicao                         VARCHAR(100),
        idade                           INT,
        qt_jogos                        INT DEFAULT 0,
        qt_jogos_titular                INT DEFAULT 0,
        minutos_jogados                 INT DEFAULT 0,
        minutos_jogados_divid_90        DECIMAL(10,2),
        gols                            INT DEFAULT 0,
        assistencias                    INT DEFAULT 0,
        participacoes_gols              INT DEFAULT 0,
        gols_nao_penalti                INT DEFAULT 0,
        gols_penalti                    INT DEFAULT 0,
        chutes_penalti                  INT DEFAULT 0,
        cartoes_amarelo                 INT DEFAULT 0,
        cartoes_vermelho                INT DEFAULT 0,
        gols_90                         DECIMAL(10,2),
        assistencias_90                 DECIMAL(10,2),
        participacoes_gols_90           DECIMAL(10,2),
        gols_nao_penalti_90             DECIMAL(10,2),
        participacoes_gols_e_penalti_90 DECIMAL(10,2)
    );
    """)

   
    # TRATAMENTO DATAFRAME
    # substitui NaN por None (compatível com MySQL)
    df = df.where(pd.notnull(df), None)

   
    # PREPARA DADOS   
    dados = [
    tuple(None if pd.isna(v) else v for v in row)
    for row in df[[
        'time','campeonato_id','nome','nacionalidade','posicao','idade',
        'qt_jogos','qt_jogos_titular','minutos_jogados',
        'minutos_jogados_divid_90','gols','assistencias',
        'participacoes_gols','gols_nao_penalti','gols_penalti',
        'chutes_penalti','cartoes_amarelo','cartoes_vermelho',
        'gols_90','assistencias_90','participacoes_gols_90',
        'gols_nao_penalti_90','participacoes_gols_e_penalti_90'
        ]].values
    ]

   
    # INSERT   
    sql = """
    INSERT INTO stats_jogadores_2025 (
        time,
        campeonato_id,
        nome,
        nacionalidade,
        posicao,
        idade,
        qt_jogos,
        qt_jogos_titular,
        minutos_jogados,
        minutos_jogados_divid_90,
        gols,
        assistencias,
        participacoes_gols,
        gols_nao_penalti,
        gols_penalti,
        chutes_penalti,
        cartoes_amarelo,
        cartoes_vermelho,
        gols_90,
        assistencias_90,
        participacoes_gols_90,
        gols_nao_penalti_90,
        participacoes_gols_e_penalti_90
    )
    VALUES (
        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s
    )
    """

    cursor.executemany(sql, dados)

    conn.commit()

    print(f"🚀 Inseridos {cursor.rowcount} registros com sucesso!")

except mysql.connector.Error as err:
    print(f"❌ Erro de Banco de Dados: {err}")

finally:
    cursor.close()
    conn.close()

✅ Após limpeza: 1125 registros válidos
   minutos_jogados
0           3060.0
1           2717.0
2           2267.0
3           2319.0
4           2050.0
float64
🚀 Inseridos 1125 registros com sucesso!


# Fim